# Probe transfer, session 2: independent evaluation set and Instruct counterparts
Runtime > Change runtime type > **A100**.

Before running: put `extract_subset.py`, `make_external.py`, `run_extra2.sh` and `eval_external.py` in the `notebooks` folder next to `extract.py`. Needs about 1.9 GB free on Drive. Nothing existing is overwritten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install transformers accelerate bitsandbytes scikit-learn scipy pandas datasets

In [ ]:
import os
from google.colab import userdata
tok = userdata.get("HF_TOKEN_READ")          # the read token in the Colab Secrets panel
os.environ["HF_TOKEN"] = tok                  # inherited by the bash cell below
os.environ["HF_HUB_DISABLE_XET"] = "1"        # use the classic downloader; the Xet one can fail on Colab
from huggingface_hub import login, whoami
login(token=tok, add_to_git_credential=False)
print("logged in as", whoami()["name"])

In [ ]:
D = "/content/drive/MyDrive/Work/Mech Interp/BlueDot Technical Project - P4/notebooks"
O = D + "/output_analysis"
import os; os.makedirs(O, exist_ok=True)
assert os.path.exists(f"{O}/xstest_prompts.csv"), "xstest_prompts.csv is expected in output_analysis"
# independent prompt set: OR-Bench hard-1k (safe), OR-Bench toxic (unsafe), 500 Alpaca instructions (safe)
!python "$D/make_external.py" --out "$O/external_prompts.csv"


## Extractions (GPU, long). Order: math-7b, qwen-14b, qwen-32b, llama-8b. If the session drops, rerun this cell; finished files are skipped.
If `meta-llama/Llama-3.1-8B-Instruct` gives 403, request access on its model page, or put `LLAMA_INST=unsloth/Llama-3.1-8B-Instruct` in front of `bash` below.

In [ ]:
!bash "$D/run_extra2.sh" "$O/xstest_prompts.csv" "$O/external_prompts.csv" "$D"


## Analysis (CPU, a few minutes). Outputs go to `output_analysis` with new names (`*_inst_*`, `*_external*`).

In [ ]:
for f, L in [("math-7b", 18), ("llama-8b", 17), ("qwen-14b", 40), ("qwen-32b", 43)]:
    print("=" * 30, f, "Instruct arm on XSTest")
    !python "$D/analyze.py" --csv "$O/xstest_prompts.csv" --base "$D/acts_{f}_base.npz" --arm "$D/acts_{f}_inst.npz" --name "$O/{f}_inst"
    !python "$D/cosines.py" --csv "$O/xstest_prompts.csv" --base "$D/acts_{f}_base.npz" --arm "$D/acts_{f}_inst.npz" --name "$O/{f}_inst"
    print("=" * 30, f, "independent set")
    !python "$D/eval_external.py" --xs "$O/xstest_prompts.csv" --ext "$O/external_prompts.csv" --layer {L} --name "$O/{f}" \
        --xs_base "$D/acts_{f}_base.npz" --xs_arm "$D/acts_{f}_arm.npz" --xs_inst "$D/acts_{f}_inst.npz" \
        --ext_base "$D/ext_{f}_base.npz" --ext_arm "$D/ext_{f}_arm.npz" --ext_inst "$D/ext_{f}_inst.npz"


Send back from `output_analysis`, for each family: `<f>_inst_folds.csv`, `<f>_inst_summary.json`, `<f>_inst_layers.csv`, `<f>_inst_cosines_summary.json`, `<f>_inst_fixedlayer_folds.csv`, `<f>_external_table.csv`, `<f>_external.json`, plus the printed output of the cell above.